# 参照訂正なしで訂正文の質を評価する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/28_gec_education/gec_reference_free.ipynb)

NLP クックブック 28-09 ／ 文法誤り訂正と言語教育

参照訂正（正解）がなくても、訂正システムの出力が「文法的か」「流暢か」「元の意味を保っているか」を自動で評価したい。

In [1]:
!pip install -q sentencepiece sentence-transformers datasets

In [2]:
import math
import numpy as np
import sentencepiece as spm
import torch
from huggingface_hub import hf_hub_download
from sentence_transformers import SentenceTransformer
from tokenizers import models
from transformers import AutoModelForCausalLM, AutoTokenizer

lm_name = "rinna/japanese-gpt2-small"
lm_tok = AutoTokenizer.from_pretrained(lm_name, use_fast=False)
# transformers 5 系の T5Tokenizer は未知語 (大文字の英字など) を文末記号 (</s>) の ID にしてしまう。
# 元の spiece.model から語彙とスコアを読み、未知語の ID を正しく扱う Unigram に差し替える
sp = spm.SentencePieceProcessor(model_file=hf_hub_download(lm_name, "spiece.model"))
lm_tok.backend_tokenizer.model = models.Unigram([(sp.id_to_piece(i), sp.get_score(i)) for i in range(sp.get_piece_size())], unk_id=sp.unk_id())
lm = AutoModelForCausalLM.from_pretrained(lm_name).eval()
embedder = SentenceTransformer("intfloat/multilingual-e5-small")

@torch.no_grad()
def fluency(sentence):
    """言語モデルによる 1 トークンあたりの対数尤度 (高いほど自然)。長さで割って、短い文が有利にならないようにする"""
    ids = lm_tok(sentence, return_tensors="pt")["input_ids"]
    return -float(lm(ids, labels=ids).loss)

def meaning(source, hypothesis):
    """元の文と訂正文の文埋め込みの類似度 (意味が保たれているか)"""
    a, b = embedder.encode(["query: " + source, "query: " + hypothesis], normalize_embeddings=True)
    return float(a @ b)

source = "私は毎日学校を行きますが、昨日は病気ので休みました。"
candidates = {
    "元の文 (訂正なし)": source,
    "正しい訂正": "私は毎日学校に行きますが、昨日は病気なので休みました。",
    "一部だけ訂正": "私は毎日学校に行きますが、昨日は病気ので休みました。",
    "流暢だが意味が違う": "私は毎日会社に行きますが、明日は休みます。",
    "意味は同じだが大きく書き換え": "毎日通学していますが、昨日は体調不良のため欠席しました。",
    "壊れた出力": "私は毎日学校行きますに昨日が、病気休みましたなので。",
}
for label, hyp in candidates.items():
    print(f"{label}: 流暢さ {fluency(hyp):.2f} / 意味の保存 {meaning(source, hyp):.3f}")

元の文 (訂正なし): 流暢さ -3.96 / 意味の保存 1.000
正しい訂正: 流暢さ -3.79 / 意味の保存 0.999
一部だけ訂正: 流暢さ -3.78 / 意味の保存 1.000
流暢だが意味が違う: 流暢さ -4.07 / 意味の保存 0.939
意味は同じだが大きく書き換え: 流暢さ -3.47 / 意味の保存 0.925
壊れた出力: 流暢さ -6.48 / 意味の保存 0.992


In [3]:
# 流暢さの「改善幅」と意味の保存を組み合わせて 1 つのスコアにする
def score(source, hypothesis, weight=0.5):
    gain = 1 / (1 + math.exp(-(fluency(hypothesis) - fluency(source)) * 2))        # 流暢さが上がれば 0.5 より大きく、下がれば小さく
    return weight * gain + (1 - weight) * meaning(source, hypothesis)

for label, hyp in sorted(candidates.items(), key=lambda x: -score(source, x[1])):
    print(f"{score(source, hyp):.3f}  {label}")

0.826  意味は同じだが大きく書き換え
0.793  一部だけ訂正
0.790  正しい訂正
0.750  元の文 (訂正なし)
0.692  流暢だが意味が違う
0.499  壊れた出力


In [4]:
# データで確かめる: 誤りを含む文より、参照訂正のほうが高いスコアになる割合
from datasets import load_dataset

ds = load_dataset("ronantakizawa/jfleg-japanese", split="validation").select(range(200))
wins = [fluency(ex["corrections"][0]) > fluency(ex["sentence"]) for ex in ds]
print("参照訂正のほうが流暢と判定された割合:", round(np.mean(wins), 3))
miss = [(ex["sentence"], ex["corrections"][0]) for ex, w in zip(ds, wins) if not w][:5]
for s, c in miss:
    print(f"   逆転した例: {s} ({fluency(s):.2f}) → {c} ({fluency(c):.2f})")

参照訂正のほうが流暢と判定された割合: 0.74
   逆転した例: 彼は公園を走っている。 (-6.82) → 彼は公園で走っている。 (-7.41)
   逆転した例: 駅で電車を乗りました。 (-5.40) → 駅で電車に乗りました。 (-5.50)
   逆転した例: 図書館に本を借りに行きます。 (-4.50) → 図書館へ本を借りに行きます。 (-4.81)
   逆転した例: 電車が駅に到着します。 (-5.37) → 電車は駅に到着します。 (-5.69)
   逆転した例: 本を五本買いました。 (-5.49) → 本を五冊買いました。 (-5.58)


- 参照訂正と比べる評価（M2、ERRANT）は、参照にない正しい訂正を誤りと数えてしまう。参照なしの評価は、訂正文そのものの質を直接見るので、この問題がない。
- 見る観点は 3 つ: 文法性（誤りが残っていないか）、流暢性（自然な文か）、意味の保存（元の文の内容を変えていないか）。1 つだけだと簡単にだませる（流暢さだけなら、無関係な自然な文が最高点になる）。
- 言語モデルの尤度は、頻度の高い語や言い回しに甘い。正しくても珍しい表現は低く、誤りを含んでいてもありふれた表現は高くなる。
- 人手評価に合うように各観点のモデルを学習した指標に SOME、訂正前後の文から品質を推定する指標に IMPARA がある。本格的に使うなら、これらの公開モデルを検討する。
- 文埋め込みの類似度は、助詞 1 つの違いのような小さな変更にはほとんど反応しない。「意味が大きく変わっていないか」の確認として使う。